# Advanced Tutorial Problems with Solutions: `json.JSONEncoder`

This notebook revisits custom JSON encoding using Python's standard-library `json` module, but in a **tutorial style**.

Instead of jumping directly to final answers, each problem is broken into small logical steps:

- inspect default behavior;
- make a first attempt;
- observe a limitation;
- refine the design;
- test the result;
- discuss the tradeoffs.

The emphasis is on building a stable **serialization contract**, not merely making `json.dumps()` stop raising exceptions.

We will cover:

- exactly when `JSONEncoder.default()` runs;
- why built-in supported values bypass `default()`;
- custom object protocols;
- tagged JSON representations;
- tag collisions and schema versions;
- safe `object_hook` decoding;
- deterministic serialization;
- timezone normalization;
- recursive preprocessing;
- path-aware validation;
- registry precedence;
- strict JSON policies;
- Unicode and compact output;
- `iterencode()`;
- contract testing.

## Setup

We only use the Python standard library in this notebook.

In [1]:
import hashlib
import json
from dataclasses import dataclass, fields, is_dataclass
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from io import StringIO
from pathlib import Path
from uuid import UUID, uuid4

# Problem 1 — What actually reaches `default()`?

Before writing an advanced encoder, we need to understand one subtle point:

> `default()` is a fallback hook. It does not see every value.

Let's prove that experimentally.

We will start with a tracing encoder. It prints every object passed to `default()` and then delegates to the standard implementation.

In [2]:
class TraceEncoder(json.JSONEncoder):
    def default(self, obj):
        print(
            "default() received:",
            type(obj).__name__,
            repr(obj),
        )
        return super().default(obj)

First, serialize only values that the standard encoder already understands.

In [3]:
data = {
    "name": "Python",
    "version": 3,
    "ratio": 3.14,
    "active": True,
    "nothing": None,
    "items": [1, 2, 3],
    "point": (10, 20),
}

print(json.dumps(data, cls=TraceEncoder))

{"name": "Python", "version": 3, "ratio": 3.14, "active": true, "nothing": null, "items": [1, 2, 3], "point": [10, 20]}


You should see no tracing output.

That tells us `default()` was not needed.

The encoder already knows how to handle:

- strings;
- integers;
- floats;
- booleans;
- `None`;
- lists;
- tuples;
- dictionaries.

Notice that a tuple becomes a JSON array automatically.

Now add a `datetime`, which is not natively JSON-serializable.

In [4]:
payload = {
    "created_at": datetime(
        2026,
        8,
        7,
        15,
        30,
        tzinfo=timezone.utc,
    )
}

try:
    json.dumps(payload, cls=TraceEncoder)
except TypeError as exc:
    print(type(exc).__name__, exc)

default() received: datetime datetime.datetime(2026, 8, 7, 15, 30, tzinfo=datetime.timezone.utc)
TypeError Object of type datetime is not JSON serializable


This time `default()` runs.

The tracing method delegates to `super().default()`, which raises `TypeError`.

That gives us the first major rule:

> Use `default()` to add support for otherwise unsupported Python objects.

Let's implement actual support for `datetime`.

In [5]:
class DateTimeEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime):
            return obj.isoformat()

        return super().default(obj)


print(json.dumps(
    {
        "created_at": datetime(
            2026,
            8,
            7,
            15,
            30,
            tzinfo=timezone.utc,
        )
    },
    cls=DateTimeEncoder,
    indent=2,
))

{
  "created_at": "2026-08-07T15:30:00+00:00"
}


### Why can't `default()` uppercase every string?

A common first attempt is to write a string branch in `default()`.

Let's try it.

In [6]:
class UppercaseStringEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, str):
            return obj.upper()

        return super().default(obj)


print(json.dumps(
    {"name": "Python"},
    cls=UppercaseStringEncoder,
))

{"name": "Python"}


Nothing changes.

The string was handled before the fallback hook was consulted.

So if already-supported values must be transformed, we need **preprocessing** rather than `default()`.

We will build that later.

# Problem 2 — Define an application-level serialization protocol

Suppose an application has many custom domain objects.

One design is to keep adding more and more `isinstance()` branches to one giant encoder.

Another design is to define a small protocol:

> If an object implements `to_json_value()`, the encoder asks the object for a JSON-compatible representation.

Let's build that step by step.

In [7]:
@dataclass(frozen=True)
class Money:
    amount: Decimal
    currency: str

    def to_json_value(self):
        return {
            "amount": str(self.amount),
            "currency": self.currency,
        }

Now create a generic protocol-aware encoder.

In [8]:
class ProtocolEncoder(json.JSONEncoder):
    def default(self, obj):
        method = getattr(
            obj,
            "to_json_value",
            None,
        )

        if callable(method):
            return method()

        return super().default(obj)

In [9]:
price = Money(
    Decimal("19.9900"),
    "EUR",
)

print(json.dumps(
    {"price": price},
    cls=ProtocolEncoder,
    indent=2,
))

{
  "price": {
    "amount": "19.9900",
    "currency": "EUR"
  }
}


This works, but after `json.loads()` the result is only a dictionary.

The original `Money` type is no longer obvious.

If round-trip reconstruction matters, we need type metadata.

Let's add an explicit tag.

In [10]:
@dataclass(frozen=True)
class TaggedMoney:
    amount: Decimal
    currency: str

    def to_json_value(self):
        return {
            "__type__": "money",
            "amount": str(self.amount),
            "currency": self.currency,
        }


print(json.dumps(
    {
        "price": TaggedMoney(
            Decimal("19.9900"),
            "EUR",
        )
    },
    cls=ProtocolEncoder,
    indent=2,
))

{
  "price": {
    "__type__": "money",
    "amount": "19.9900",
    "currency": "EUR"
  }
}


That is more informative, but it introduces a new problem:

What if normal application data already contains a key named `__type__`?

That is a **tag collision** problem.

# Problem 3 — Avoid collisions between tags and user data

Consider this completely ordinary dictionary:

In [11]:
ordinary_data = {
    "__type__": "money",
    "amount": "this is just user text",
}

print(json.dumps(ordinary_data, indent=2))

{
  "__type__": "money",
  "amount": "this is just user text"
}


A careless decoder might incorrectly convert that dictionary into a `Money` object.

One way to reduce this ambiguity is to reserve a dedicated metadata envelope.

For this notebook, we will use:

```json
{
  "$python": {
    "type": "money",
    "version": 1
  },
  "data": {
    "amount": "19.99",
    "currency": "EUR"
  }
}
```

The exact key name is not magical. The important part is that the metadata convention is deliberate and documented.

In [12]:
@dataclass(frozen=True)
class NamespacedMoney:
    amount: Decimal
    currency: str

    def to_json_value(self):
        return {
            "$python": {
                "type": "money",
                "version": 1,
            },
            "data": {
                "amount": str(self.amount),
                "currency": self.currency,
            },
        }


value = NamespacedMoney(
    Decimal("123.4500"),
    "USD",
)

print(json.dumps(
    {"value": value},
    cls=ProtocolEncoder,
    indent=2,
))

{
  "value": {
    "$python": {
      "type": "money",
      "version": 1
    },
    "data": {
      "amount": "123.4500",
      "currency": "USD"
    }
  }
}


This representation is larger, but it makes three things explicit:

1. the object is serializer-managed;
2. the logical type is `money`;
3. the schema version is `1`.

That version field becomes important when formats evolve.

# Problem 4 — Build a safe tagged decoder

Encoding is only half the problem.

Python's `json.loads()` supports `object_hook`, which receives each decoded JSON object.

Let's begin with a simple decoder.

In [13]:
def simple_object_hook(obj):
    metadata = obj.get("$python")

    if metadata is None:
        return obj

    if metadata.get("type") == "money":
        data = obj["data"]

        return Money(
            amount=Decimal(data["amount"]),
            currency=data["currency"],
        )

    return obj


text = json.dumps(
    {
        "price": NamespacedMoney(
            Decimal("19.99"),
            "EUR",
        )
    },
    cls=ProtocolEncoder,
)

decoded = json.loads(
    text,
    object_hook=simple_object_hook,
)

print(decoded)
print(type(decoded["price"]))

{'price': Money(amount=Decimal('19.99'), currency='EUR')}
<class '__main__.Money'>


This works for valid input, but a robust decoder has to think about malformed data.

Questions to ask:

- Is `$python` actually an object?
- Is the tag recognized?
- Is the version supported?
- Is `data` present?
- Are required fields present?
- Can the amount be parsed?
- Is the currency really a string?

Let's make those decisions explicit.

In [14]:
def strict_object_hook(obj):
    metadata = obj.get("$python")

    if metadata is None:
        return obj

    if not isinstance(metadata, dict):
        raise ValueError(
            "$python metadata must be an object"
        )

    tag = metadata.get("type")
    version = metadata.get("version")

    if tag != "money":
        return obj

    if version != 1:
        raise ValueError(
            f"Unsupported money version: {version!r}"
        )

    data = obj.get("data")

    if not isinstance(data, dict):
        raise ValueError(
            "money.data must be an object"
        )

    if "amount" not in data:
        raise ValueError(
            "money.data.amount is required"
        )

    if "currency" not in data:
        raise ValueError(
            "money.data.currency is required"
        )

    try:
        amount = Decimal(data["amount"])
    except Exception as exc:
        raise ValueError(
            "money.data.amount is invalid"
        ) from exc

    currency = data["currency"]

    if not isinstance(currency, str):
        raise ValueError(
            "money.data.currency must be a string"
        )

    return Money(
        amount=amount,
        currency=currency,
    )

Let's test a valid payload.

In [15]:
valid = {
    "$python": {
        "type": "money",
        "version": 1,
    },
    "data": {
        "amount": "12.50",
        "currency": "EUR",
    },
}

print(json.loads(
    json.dumps(valid),
    object_hook=strict_object_hook,
))

Money(amount=Decimal('12.50'), currency='EUR')


Now test an unsupported schema version.

In [16]:
invalid_version = {
    "$python": {
        "type": "money",
        "version": 999,
    },
    "data": {
        "amount": "12.50",
        "currency": "EUR",
    },
}

try:
    json.loads(
        json.dumps(invalid_version),
        object_hook=strict_object_hook,
    )
except ValueError as exc:
    print(type(exc).__name__, exc)

ValueError Unsupported money version: 999


### Security lesson

A decoder should normally use an **allowlist** of recognized tags.

Avoid reading arbitrary class names from JSON and dynamically importing or instantiating those classes.

Untrusted JSON should not be allowed to choose executable Python code.

# Problem 5 — Support schema migrations

Long-lived JSON formats change.

Suppose version 1 stores money like this:

```json
{
  "$python": {"type": "money", "version": 1},
  "data": {"amount": "12.50", "currency": "EUR"}
}
```

Later, version 2 stores integer minor units:

```json
{
  "$python": {"type": "money", "version": 2},
  "data": {"minor_units": 1250, "scale": 2, "currency": "EUR"}
}
```

We want both versions to decode to the same modern `Money` object.

A clean design keeps each version-specific migration small and independently testable.

In [17]:
def decode_money_v1(data):
    return Money(
        amount=Decimal(data["amount"]),
        currency=data["currency"],
    )


def decode_money_v2(data):
    minor_units = int(data["minor_units"])
    scale = int(data["scale"])

    amount = (
        Decimal(minor_units)
        / (Decimal(10) ** scale)
    )

    return Money(
        amount=amount,
        currency=data["currency"],
    )

Now the object hook only needs to choose the appropriate decoder.

In [18]:
def versioned_object_hook(obj):
    metadata = obj.get("$python")

    if not isinstance(metadata, dict):
        return obj

    if metadata.get("type") != "money":
        return obj

    version = metadata.get("version")
    data = obj.get("data")

    if not isinstance(data, dict):
        raise ValueError(
            "money.data must be an object"
        )

    if version == 1:
        return decode_money_v1(data)

    if version == 2:
        return decode_money_v2(data)

    raise ValueError(
        f"Unsupported money version: {version!r}"
    )

In [19]:
v1 = {
    "$python": {
        "type": "money",
        "version": 1,
    },
    "data": {
        "amount": "12.50",
        "currency": "EUR",
    },
}

v2 = {
    "$python": {
        "type": "money",
        "version": 2,
    },
    "data": {
        "minor_units": 1250,
        "scale": 2,
        "currency": "EUR",
    },
}

money_1 = json.loads(
    json.dumps(v1),
    object_hook=versioned_object_hook,
)

money_2 = json.loads(
    json.dumps(v2),
    object_hook=versioned_object_hook,
)

print(money_1)
print(money_2)

assert money_1 == money_2

Money(amount=Decimal('12.50'), currency='EUR')
Money(amount=Decimal('12.5'), currency='EUR')


The important design point is that the **data schema version** is separate from the Python package version.

A serializer may need to read old data long after the code has changed.

# Problem 6 — Produce deterministic JSON for cache keys

Suppose we want to hash JSON text to create cache keys.

These dictionaries are logically equal:

In [20]:
a = {
    "name": "Ada",
    "age": 36,
}

b = {
    "age": 36,
    "name": "Ada",
}

print(a == b)

True


But ordinary JSON output can differ because key order is preserved from the Python dictionary.

In [21]:
text_a = json.dumps(a)
text_b = json.dumps(b)

print(text_a)
print(text_b)
print(text_a == text_b)

{"name": "Ada", "age": 36}
{"age": 36, "name": "Ada"}
False


For a stable representation, we can sort keys and remove insignificant whitespace.

In [22]:
def deterministic_json(obj):
    return json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    )


stable_a = deterministic_json(a)
stable_b = deterministic_json(b)

print(stable_a)
print(stable_b)

assert stable_a == stable_b

{"age":36,"name":"Ada"}
{"age":36,"name":"Ada"}


Now hash the UTF-8 representation.

In [23]:
def json_cache_key(obj):
    text = deterministic_json(obj)

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


print(json_cache_key(a))
print(json_cache_key(b))

assert json_cache_key(a) == json_cache_key(b)

fc234b36d9984d8c111697eae4b5315ba69588f24c68ee52a1f78c2ea5969d8f
fc234b36d9984d8c111697eae4b5315ba69588f24c68ee52a1f78c2ea5969d8f


This is useful for many application cache keys and regression snapshots.

But stable stdlib output should not automatically be treated as a formal canonical JSON standard for cryptographic protocols.

# Problem 7 — Normalize equivalent datetimes before deterministic serialization

Two timezone-aware datetimes can represent the same instant.

In [24]:
utc_time = datetime(
    2026,
    8,
    7,
    12,
    0,
    tzinfo=timezone.utc,
)

offset_time = datetime(
    2026,
    8,
    7,
    14,
    0,
    tzinfo=timezone(
        timedelta(hours=2)
    ),
)

print(utc_time == offset_time)
print(utc_time.isoformat())
print(offset_time.isoformat())

True
2026-08-07T12:00:00+00:00
2026-08-07T14:00:00+02:00


They compare equal, but their ISO strings differ.

If those strings become cache-key input, equal instants could produce different hashes.

Let's define a stronger policy:

- naive datetimes are rejected;
- aware datetimes are normalized to UTC.

In [25]:
class UTCNormalizingEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime):
            if (
                obj.tzinfo is None
                or obj.utcoffset() is None
            ):
                raise ValueError(
                    "Naive datetime is not allowed"
                )

            normalized = obj.astimezone(
                timezone.utc
            )

            return normalized.isoformat()

        return super().default(obj)


text_1 = json.dumps(
    {"time": utc_time},
    cls=UTCNormalizingEncoder,
    sort_keys=True,
    separators=(",", ":"),
)

text_2 = json.dumps(
    {"time": offset_time},
    cls=UTCNormalizingEncoder,
    sort_keys=True,
    separators=(",", ":"),
)

print(text_1)
print(text_2)

assert text_1 == text_2

{"time":"2026-08-07T12:00:00+00:00"}
{"time":"2026-08-07T12:00:00+00:00"}


This is a subtle but important example:

serialization policy can affect equality, identity, hashing, deduplication, and testing.

# Problem 8 — Transform supported strings with preprocessing

We already proved that `default()` cannot rewrite ordinary strings.

Suppose our new requirement is:

- trim whitespace from every string value;
- recurse through lists and tuples;
- recurse through dictionaries;
- leave dictionary keys unchanged.

That is a preprocessing task.

In [26]:
def normalize_strings(obj):
    if isinstance(obj, str):
        return obj.strip()

    if isinstance(obj, list):
        return [
            normalize_strings(item)
            for item in obj
        ]

    if isinstance(obj, tuple):
        return [
            normalize_strings(item)
            for item in obj
        ]

    if isinstance(obj, dict):
        return {
            key: normalize_strings(value)
            for key, value in obj.items()
        }

    return obj


raw = {
    "name": "  Ada  ",
    "tags": [
        " python ",
        " json",
        "testing ",
    ],
    "nested": {
        "message": " hello world ",
    },
}

clean = normalize_strings(raw)

print(json.dumps(
    clean,
    indent=2,
))

{
  "name": "Ada",
  "tags": [
    "python",
    "json",
    "testing"
  ],
  "nested": {
    "message": "hello world"
  }
}


This works, but transformations can become difficult to debug.

A more advanced version can track the path of every changed value.

In [27]:
def normalize_strings_with_log(
    obj,
    path="$",
    changes=None,
):
    if changes is None:
        changes = []

    if isinstance(obj, str):
        normalized = obj.strip()

        if normalized != obj:
            changes.append(
                (path, obj, normalized)
            )

        return normalized

    if isinstance(obj, list):
        return [
            normalize_strings_with_log(
                item,
                path=f"{path}[{index}]",
                changes=changes,
            )
            for index, item in enumerate(obj)
        ]

    if isinstance(obj, tuple):
        return [
            normalize_strings_with_log(
                item,
                path=f"{path}[{index}]",
                changes=changes,
            )
            for index, item in enumerate(obj)
        ]

    if isinstance(obj, dict):
        return {
            key: normalize_strings_with_log(
                value,
                path=f"{path}.{key}",
                changes=changes,
            )
            for key, value in obj.items()
        }

    return obj


changes = []

clean = normalize_strings_with_log(
    raw,
    changes=changes,
)

print(json.dumps(clean, indent=2))

print()
print("Changes:")

for path, before, after in changes:
    print(
        path,
        repr(before),
        "->",
        repr(after),
    )

{
  "name": "Ada",
  "tags": [
    "python",
    "json",
    "testing"
  ],
  "nested": {
    "message": "hello world"
  }
}

Changes:
$.name '  Ada  ' -> 'Ada'
$.tags[0] ' python ' -> 'python'
$.tags[1] ' json' -> 'json'
$.tags[2] 'testing ' -> 'testing'
$.nested.message ' hello world ' -> 'hello world'


The same recursive pattern can support:

- redaction;
- validation;
- normalization;
- canonicalization;
- conversion;
- diagnostics.

# Problem 9 — Report the path of unsupported objects

A normal serialization error may say:

```text
Object of type X is not JSON serializable
```

In a deeply nested structure, that does not always tell us where the bad value lives.

Let's build a preflight validator.

In [28]:
CUSTOM_ALLOWED_TYPES = (
    datetime,
    Decimal,
    UUID,
    Path,
)


def find_unsupported(
    obj,
    path="$",
    problems=None,
):
    if problems is None:
        problems = []

    if obj is None:
        return problems

    if isinstance(
        obj,
        (str, int, float, bool),
    ):
        return problems

    if isinstance(
        obj,
        CUSTOM_ALLOWED_TYPES,
    ):
        return problems

    if isinstance(
        obj,
        (list, tuple),
    ):
        for index, item in enumerate(obj):
            find_unsupported(
                item,
                path=f"{path}[{index}]",
                problems=problems,
            )

        return problems

    if isinstance(obj, dict):
        for key, value in obj.items():
            find_unsupported(
                value,
                path=f"{path}.{key}",
                problems=problems,
            )

        return problems

    problems.append(
        (
            path,
            type(obj).__name__,
        )
    )

    return problems

Now test the validator with two unsupported objects.

In [29]:
class SocketLikeThing:
    pass


bad_payload = {
    "user": {
        "name": "Ada",
        "session": SocketLikeThing(),
    },
    "items": [
        1,
        2,
        {"bad": object()},
    ],
}

problems = find_unsupported(
    bad_payload
)

for path, type_name in problems:
    print(
        f"{path}: unsupported {type_name}"
    )

$.user.session: unsupported SocketLikeThing
$.items[2].bad: unsupported object


A preflight validation pass is useful when we want to collect multiple problems before writing any output.

This is especially helpful in APIs, batch jobs, and configuration generation.

# Problem 10 — Registry precedence with inheritance

Registry-based encoders can be elegant, but inheritance creates subtle matching problems.

A `datetime` is also a `date`.

In [30]:
print(
    isinstance(
        datetime.now(),
        date,
    )
)

True


Suppose a registry uses `isinstance()` and the first matching rule wins.

If `date` is registered before `datetime`, a `datetime` may be encoded using the broader rule.

In [31]:
class OrderedRegistryEncoder(json.JSONEncoder):
    registry = []

    @classmethod
    def register(
        cls,
        python_type,
        serializer,
    ):
        cls.registry.append(
            (
                python_type,
                serializer,
            )
        )

    def default(self, obj):
        for python_type, serializer in self.registry:
            if isinstance(
                obj,
                python_type,
            ):
                return serializer(obj)

        return super().default(obj)


class BadTemporalRegistryEncoder(
    OrderedRegistryEncoder
):
    registry = []


BadTemporalRegistryEncoder.register(
    date,
    lambda value: {
        "kind": "date",
        "value": value.isoformat(),
    },
)

BadTemporalRegistryEncoder.register(
    datetime,
    lambda value: {
        "kind": "datetime",
        "value": value.isoformat(),
    },
)


value = datetime(
    2026,
    8,
    7,
    12,
    0,
)

print(json.dumps(
    {"value": value},
    cls=BadTemporalRegistryEncoder,
    indent=2,
))

{
  "value": {
    "kind": "date",
    "value": "2026-08-07T12:00:00"
  }
}


The `datetime` is captured by the `date` rule.

There are several possible solutions.

One is to register more specific types first.

Another is to prefer exact type matches before broader subclass matches.

In [32]:
class ExactFirstRegistryEncoder(
    json.JSONEncoder
):
    registry = {}

    @classmethod
    def register(
        cls,
        python_type,
        serializer,
    ):
        cls.registry[
            python_type
        ] = serializer

    def default(self, obj):
        exact = self.registry.get(
            type(obj)
        )

        if exact is not None:
            return exact(obj)

        for python_type, serializer in self.registry.items():
            if isinstance(
                obj,
                python_type,
            ):
                return serializer(obj)

        return super().default(obj)

The exact-first policy gives us an explicit precedence order:

1. exact type;
2. inherited match;
3. base encoder fallback.

That is much easier to reason about than accidental registration order.

# Problem 11 — Reject duplicate serializer registration

In a large application, serializer registrations may come from many modules.

Silently replacing an existing serializer can unexpectedly change persisted or transmitted data.

Let's fail loudly instead.

In [33]:
class SafeRegistryEncoder(json.JSONEncoder):
    registry = {}

    @classmethod
    def register(
        cls,
        python_type,
        serializer,
    ):
        if python_type in cls.registry:
            raise ValueError(
                "Serializer already registered "
                f"for {python_type.__name__}"
            )

        cls.registry[
            python_type
        ] = serializer

    def default(self, obj):
        serializer = self.registry.get(
            type(obj)
        )

        if serializer is not None:
            return serializer(obj)

        return super().default(obj)


SafeRegistryEncoder.register(
    Decimal,
    lambda value: str(value),
)

try:
    SafeRegistryEncoder.register(
        Decimal,
        lambda value: float(value),
    )
except ValueError as exc:
    print(type(exc).__name__, exc)

ValueError Serializer already registered for Decimal


This follows a useful serialization principle:

> Prefer an explicit configuration error over silently changing the data contract.

# Problem 12 — Build a realistic API encoder

Let's combine several ideas into one application encoder.

Requirements:

- datetimes must be timezone-aware;
- datetimes are normalized to UTC;
- `Decimal` preserves exact precision;
- `UUID` has an explicit tag;
- dataclasses serialize declared fields only;
- Unicode remains readable;
- non-finite floats are rejected;
- unsupported custom objects still fail.

In [34]:
@dataclass(frozen=True)
class Customer:
    customer_id: UUID
    name: str


@dataclass(frozen=True)
class Invoice:
    invoice_id: UUID
    customer: Customer
    total: Decimal
    issued_at: datetime

Now define the encoder.

In [35]:
class APIJSONEncoder(json.JSONEncoder):
    def __init__(
        self,
        *args,
        **kwargs,
    ):
        kwargs["ensure_ascii"] = False
        kwargs["allow_nan"] = False

        super().__init__(
            *args,
            **kwargs,
        )

    def default(self, obj):
        if isinstance(obj, datetime):
            if (
                obj.tzinfo is None
                or obj.utcoffset() is None
            ):
                raise ValueError(
                    "Naive datetime is not allowed"
                )

            normalized = obj.astimezone(
                timezone.utc
            )

            return {
                "$python": {
                    "type": "datetime",
                    "version": 1,
                },
                "data": {
                    "value": normalized.isoformat()
                },
            }

        if isinstance(obj, Decimal):
            return {
                "$python": {
                    "type": "decimal",
                    "version": 1,
                },
                "data": {
                    "value": str(obj)
                },
            }

        if isinstance(obj, UUID):
            return {
                "$python": {
                    "type": "uuid",
                    "version": 1,
                },
                "data": {
                    "value": str(obj)
                },
            }

        if (
            is_dataclass(obj)
            and not isinstance(obj, type)
        ):
            return {
                "$python": {
                    "type": "dataclass",
                    "version": 1,
                    "name": type(obj).__name__,
                },
                "data": {
                    field.name: getattr(
                        obj,
                        field.name,
                    )
                    for field in fields(obj)
                },
            }

        return super().default(obj)

Create a nested invoice with Unicode text, a high-precision decimal, and a non-UTC timezone.

In [36]:
invoice = Invoice(
    invoice_id=uuid4(),
    customer=Customer(
        customer_id=uuid4(),
        name="Ада Лъвлейс",
    ),
    total=Decimal(
        "1234.5600000000000000001"
    ),
    issued_at=datetime(
        2026,
        8,
        7,
        18,
        30,
        tzinfo=timezone(
            timedelta(hours=3)
        ),
    ),
)

Serialize it with indentation so the tagged structure is easy to inspect.

In [37]:
api_text = json.dumps(
    invoice,
    cls=APIJSONEncoder,
    indent=2,
    sort_keys=True,
)

print(api_text)

{
  "$python": {
    "name": "Invoice",
    "type": "dataclass",
    "version": 1
  },
  "data": {
    "customer": {
      "$python": {
        "name": "Customer",
        "type": "dataclass",
        "version": 1
      },
      "data": {
        "customer_id": {
          "$python": {
            "type": "uuid",
            "version": 1
          },
          "data": {
            "value": "00b4c893-ada7-4550-9dc9-b5370fade275"
          }
        },
        "name": "Ада Лъвлейс"
      }
    },
    "invoice_id": {
      "$python": {
        "type": "uuid",
        "version": 1
      },
      "data": {
        "value": "0ca9042c-1514-4222-b43b-5ecb403d0ac5"
      }
    },
    "issued_at": {
      "$python": {
        "type": "datetime",
        "version": 1
      },
      "data": {
        "value": "2026-08-07T15:30:00+00:00"
      }
    },
    "total": {
      "$python": {
        "type": "decimal",
        "version": 1
      },
      "data": {
        "value": "1234.56000000000000000

Now verify the important policy guarantees.

In [38]:
assert "Ада Лъвлейс" in api_text

assert (
    "1234.5600000000000000001"
    in api_text
)

assert "+00:00" in api_text

json.loads(api_text)

print(
    "API serialization checks passed."
)

API serialization checks passed.


The dataclass branch serializes declared dataclass fields rather than blindly exposing `obj.__dict__`.

That reduces accidental leakage of caches, private attributes, and implementation details.

# Problem 13 — Reconstruct only allowlisted dataclasses

To decode primitive tagged values, we can safely use known constructors.

For dataclasses, we should not accept arbitrary class names from JSON.

Instead, define an allowlist.

In [39]:
SAFE_DATACLASSES = {
    "Customer": Customer,
    "Invoice": Invoice,
}

Now create the matching `object_hook`.

In [40]:
def api_object_hook(obj):
    metadata = obj.get("$python")

    if not isinstance(metadata, dict):
        return obj

    tag = metadata.get("type")
    version = metadata.get("version")
    data = obj.get("data")

    if version != 1:
        raise ValueError(
            f"Unsupported version: {version!r}"
        )

    if tag == "datetime":
        return datetime.fromisoformat(
            data["value"]
        )

    if tag == "decimal":
        return Decimal(
            data["value"]
        )

    if tag == "uuid":
        return UUID(
            data["value"]
        )

    if tag == "dataclass":
        name = metadata.get("name")
        cls = SAFE_DATACLASSES.get(name)

        if cls is None:
            raise ValueError(
                f"Unsupported dataclass: {name!r}"
            )

        return cls(**data)

    return obj

Because `object_hook` processes nested objects from the inside out, custom fields are reconstructed before the outer dataclass is created.

Let's verify the complete round trip.

In [41]:
restored = json.loads(
    api_text,
    object_hook=api_object_hook,
)

print(restored)
print(type(restored))

assert restored == invoice

Invoice(invoice_id=UUID('0ca9042c-1514-4222-b43b-5ecb403d0ac5'), customer=Customer(customer_id=UUID('00b4c893-ada7-4550-9dc9-b5370fade275'), name='Ада Лъвлейс'), total=Decimal('1234.5600000000000000001'), issued_at=datetime.datetime(2026, 8, 7, 15, 30, tzinfo=datetime.timezone.utc))
<class '__main__.Invoice'>


We now have a complete contract:

Python objects → tagged JSON → Python objects.

# Problem 14 — Pretty JSON vs compact JSON

Custom type representation and text formatting are separate concerns.

Let's serialize the same object twice.

In [42]:
pretty_json = json.dumps(
    invoice,
    cls=APIJSONEncoder,
    indent=2,
    sort_keys=True,
)

compact_json = json.dumps(
    invoice,
    cls=APIJSONEncoder,
    sort_keys=True,
    separators=(",", ":"),
)

pretty_bytes = len(
    pretty_json.encode("utf-8")
)

compact_bytes = len(
    compact_json.encode("utf-8")
)

print("Pretty bytes :", pretty_bytes)
print("Compact bytes:", compact_bytes)
print(
    "Saved bytes  :",
    pretty_bytes - compact_bytes,
)

Pretty bytes : 1033
Compact bytes: 599
Saved bytes  : 434


Pretty JSON is useful for people.

Compact JSON is useful for transport and storage.

The same custom type mapping should work with either formatting style.

# Problem 15 — Unicode output

By default, `json.dumps()` uses `ensure_ascii=True`.

Let's compare escaped and readable Unicode output.

In [43]:
message = {
    "text": "Здравей 🌍 — café"
}

ascii_json = json.dumps(
    message,
    ensure_ascii=True,
)

unicode_json = json.dumps(
    message,
    ensure_ascii=False,
)

print("ASCII escaped:")
print(ascii_json)

print()
print("Readable Unicode:")
print(unicode_json)

ASCII escaped:
{"text": "\u0417\u0434\u0440\u0430\u0432\u0435\u0439 \ud83c\udf0d \u2014 caf\u00e9"}

Readable Unicode:
{"text": "Здравей 🌍 — café"}


Now compare UTF-8 byte lengths.

In [44]:
print(
    "ASCII bytes:",
    len(ascii_json.encode("utf-8")),
)

print(
    "Unicode bytes:",
    len(unicode_json.encode("utf-8")),
)

ASCII bytes: 84
Unicode bytes: 41


For modern UTF-8 systems, `ensure_ascii=False` is often easier to read and may even be smaller.

The final choice should match the requirements of the surrounding protocol.

# Problem 16 — Stream output with `iterencode()`

`encode()` returns one complete string.

`iterencode()` yields chunks incrementally.

Let's inspect those chunks.

In [45]:
encoder = APIJSONEncoder(
    sort_keys=True,
    separators=(",", ":"),
)

chunks = list(
    encoder.iterencode(invoice)
)

print(
    "Number of chunks:",
    len(chunks),
)

for chunk in chunks[:20]:
    print(repr(chunk))

Number of chunks: 145
'{'
'"$python"'
':'
'{'
'"name"'
':'
'"Invoice"'
','
'"type"'
':'
'"dataclass"'
','
'"version"'
':'
'1'
'}'
','
'"data"'
':'
'{'


Now write the chunks into a file-like object.

In [46]:
buffer = StringIO()

for chunk in encoder.iterencode(invoice):
    buffer.write(chunk)

streamed = buffer.getvalue()

print(streamed[:300])

{"$python":{"name":"Invoice","type":"dataclass","version":1},"data":{"customer":{"$python":{"name":"Customer","type":"dataclass","version":1},"data":{"customer_id":{"$python":{"type":"uuid","version":1},"data":{"value":"00b4c893-ada7-4550-9dc9-b5370fade275"}},"name":"Ада Лъвлейс"}},"invoice_id":{"$p


Finally, decode the streamed representation and verify the round trip.

In [47]:
round_trip = json.loads(
    streamed,
    object_hook=api_object_hook,
)

assert round_trip == invoice

print(
    "Streaming round-trip succeeded."
)

Streaming round-trip succeeded.


`iterencode()` is useful when a destination can consume data incrementally:

- files;
- sockets;
- compressed streams;
- HTTP response bodies;
- custom transport layers.

But remember: if `default()` converts a huge object into a giant list, much of the memory benefit can disappear.

# Problem 17 — Detect JSON key collisions

Python dictionary keys can be more permissive than JSON object names.

Consider these two distinct Python keys:

In [48]:
collision_data = {
    "1": "string one",
    1: "integer one",
}

print(collision_data)
print(json.dumps(collision_data))

{'1': 'string one', 1: 'integer one'}
{"1": "string one", "1": "integer one"}


The JSON representation uses string object member names, so semantic distinctions can collapse.

A stricter application can normalize keys explicitly and reject collisions.

In [49]:
def normalize_json_keys(mapping):
    result = {}

    for key, value in mapping.items():
        if isinstance(key, str):
            normalized = key

        elif isinstance(key, bool):
            normalized = (
                "true"
                if key
                else "false"
            )

        elif key is None:
            normalized = "null"

        elif isinstance(
            key,
            (int, float),
        ):
            normalized = str(key)

        else:
            raise TypeError(
                "Unsupported key type: "
                f"{type(key).__name__}"
            )

        if normalized in result:
            raise ValueError(
                "JSON key collision for "
                f"{normalized!r}"
            )

        result[normalized] = value

    return result


try:
    normalize_json_keys(collision_data)
except ValueError as exc:
    print(type(exc).__name__, exc)

ValueError JSON key collision for '1'


This is a useful reminder:

> Successful serialization does not automatically mean the representation preserved every distinction in the original Python data.

# Problem 18 — Centralize application serialization policy

If every call site manually repeats:

- the custom encoder;
- Unicode settings;
- strict numeric settings;
- key ordering;

eventually someone will forget an option.

A wrapper function can centralize the contract.

In [50]:
def api_dumps(
    obj,
    **kwargs,
):
    options = {
        "cls": APIJSONEncoder,
        "ensure_ascii": False,
        "allow_nan": False,
        "sort_keys": True,
    }

    options.update(kwargs)

    return json.dumps(
        obj,
        **options,
    )


print(
    api_dumps(
        invoice,
        indent=2,
    )[:500]
)

{
  "$python": {
    "name": "Invoice",
    "type": "dataclass",
    "version": 1
  },
  "data": {
    "customer": {
      "$python": {
        "name": "Customer",
        "type": "dataclass",
        "version": 1
      },
      "data": {
        "customer_id": {
          "$python": {
            "type": "uuid",
            "version": 1
          },
          "data": {
            "value": "00b4c893-ada7-4550-9dc9-b5370fade275"
          }
        },
        "name": "Ада Лъвлейс"
      }
    },


Now callers can focus on formatting rather than remembering policy.

For example, a compact transport call becomes:

In [51]:
print(
    api_dumps(
        invoice,
        separators=(",", ":"),
    )[:300]
)

{"$python":{"name":"Invoice","type":"dataclass","version":1},"data":{"customer":{"$python":{"name":"Customer","type":"dataclass","version":1},"data":{"customer_id":{"$python":{"type":"uuid","version":1},"data":{"value":"00b4c893-ada7-4550-9dc9-b5370fade275"}},"name":"Ада Лъвлейс"}},"invoice_id":{"$p


This separates three concerns:

- application serialization policy;
- custom type representation;
- text presentation.

That usually makes serializer behavior easier to audit.

# Problem 19 — Test the serialization contract

A serializer is part of your data interface.

Tests should focus on observable guarantees, not implementation trivia.

Let's test:

- decimal precision;
- timezone normalization;
- naive datetime rejection;
- non-finite float rejection;
- readable Unicode;
- full round-trip reconstruction.

In [52]:
def test_decimal_precision():
    value = Decimal(
        "0.12345678901234567890123456789"
    )

    text = api_dumps(
        {"value": value}
    )

    assert str(value) in text


def test_datetime_normalized_to_utc():
    value = datetime(
        2026,
        8,
        7,
        15,
        0,
        tzinfo=timezone(
            timedelta(hours=3)
        ),
    )

    text = api_dumps(
        {"value": value}
    )

    assert "12:00:00+00:00" in text


def test_naive_datetime_rejected():
    value = datetime(
        2026,
        8,
        7,
        15,
        0,
    )

    try:
        api_dumps(
            {"value": value}
        )
    except ValueError:
        return

    raise AssertionError(
        "Naive datetime was not rejected"
    )


def test_non_finite_float_rejected():
    try:
        api_dumps(
            {"value": float("inf")}
        )
    except ValueError:
        return

    raise AssertionError(
        "Infinity was not rejected"
    )


def test_unicode_readable():
    text = api_dumps(
        {"value": "Здравей 🌍"}
    )

    assert "Здравей" in text
    assert "🌍" in text


def test_invoice_round_trip():
    text = api_dumps(invoice)

    restored = json.loads(
        text,
        object_hook=api_object_hook,
    )

    assert restored == invoice

In [53]:
tests = [
    test_decimal_precision,
    test_datetime_normalized_to_utc,
    test_naive_datetime_rejected,
    test_non_finite_float_rejected,
    test_unicode_readable,
    test_invoice_round_trip,
]

for test in tests:
    test()
    print(
        test.__name__,
        "PASSED",
    )

test_decimal_precision PASSED
test_datetime_normalized_to_utc PASSED
test_naive_datetime_rejected PASSED
test_non_finite_float_rejected PASSED
test_unicode_readable PASSED
test_invoice_round_trip PASSED


These tests verify the external contract:

- precision;
- timezone semantics;
- strictness;
- Unicode;
- reversibility.

That is more durable than testing that a particular internal branch ran.

# Problem 20 — Design review: which technique belongs where?

We used several different customization layers.

It is worth summarizing their roles.

## Use `default()` when:

- a Python value is not natively serializable;
- you want to map a custom type into normal JSON-compatible data;
- you do not need to rewrite built-in scalar/container behavior.

## Use preprocessing when:

- strings, integers, lists, tuples, or dictionaries must be transformed;
- you need redaction or normalization;
- you need path-aware diagnostics;
- you need to detect key collisions before encoding.

## Use a wrapper function when:

- the application has standard encoder settings;
- call sites should not repeat policy;
- you want one place to enforce strict JSON behavior.

## Use tagged objects when:

- round-trip reconstruction matters;
- type information would otherwise be ambiguous;
- persisted schemas may evolve.

## Use schema versions when:

- data can outlive one release of the application;
- migrations must be explicit;
- older representations still need to decode safely.

# Final Challenge Set

These problems intentionally leave more design work to you.

## Challenge 1 — Add `frozenset`

Create a tagged representation that round-trips a `frozenset` without silently turning it into an ordinary `set` or `list`.

## Challenge 2 — Add `Path`

Encode `pathlib.Path` using a type tag.

Then decide whether decoding should recreate a `Path` or keep a string for cross-platform portability.

## Challenge 3 — Find the exact path of a bad float

Before serialization, recursively locate `NaN`, `Infinity`, and `-Infinity`.

Report paths such as:

```text
$.metrics[4].ratio
```

## Challenge 4 — Add version 3 of `Money`

Version 3 should store `minor_units` as a string:

```json
{
  "$python": {"type": "money", "version": 3},
  "data": {
    "minor_units": "1250",
    "scale": 2,
    "currency": "EUR"
  }
}
```

Support versions 1, 2, and 3 in one decoder.

## Challenge 5 — Priority-based registry dispatch

Extend the registry so every serializer has a numeric priority.

If multiple `isinstance()` rules match, the highest priority wins.

Test it with `date` and `datetime`.

## Challenge 6 — Maximum iterator size

Support arbitrary iterators, but refuse to materialize more than 10,000 values.

Raise a clear exception when the limit is exceeded.

## Challenge 7 — JSON Lines

Write a sequence of invoices as JSON Lines:

- one JSON object per line;
- each line uses `APIJSONEncoder`;
- decode the file one line at a time.

## Challenge 8 — Compare `default=` and `cls=`

Implement identical `Decimal` and `datetime` support twice:

- once with a `default` function;
- once with a `JSONEncoder` subclass.

Compare readability, configurability, and testability.

## Challenge 9 — Reserved metadata collisions

Design a strategy for ordinary user dictionaries that contain the reserved `$python` key.

Possible approaches include:

- rejection;
- escaping;
- wrapping normal dictionaries.

Document the tradeoff.

## Challenge 10 — Write the serialization contract

For every custom type, document:

- type tag;
- current schema version;
- required fields;
- optional fields;
- precision rules;
- timezone rules;
- failure behavior;
- backward compatibility guarantees.

# Final Summary

The most important ideas from this tutorial are:

1. `default()` is a fallback for unsupported values.
2. Built-in supported values require preprocessing when they must be transformed.
3. Tagged representations preserve type information.
4. Metadata keys need a collision strategy.
5. Decoders should validate tags and versions explicitly.
6. Schema migrations should be isolated and tested.
7. Deterministic JSON requires deliberate normalization.
8. Timezone normalization matters when JSON participates in hashing or identity.
9. Registry precedence must account for inheritance.
10. Application-wide serializer policy should be centralized.
11. Contract tests should verify externally visible guarantees.
12. Successful encoding is only useful if the JSON preserves the intended meaning.